In [2]:
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
import sys
import os

# add the project folder to Python path
sys.path.append(r"C:\Users\nigel\Desktop\prax\robust-anomaly-cifar10")



def gaussian_noise(img, severity=1):
    """
    severity: 1 (low) to 5 (high)
    """
    levels = [0.02, 0.05, 0.08, 0.12, 0.18]
    std = levels[severity - 1]

    x = np.array(img).astype(np.float32) / 255.0
    noise = np.random.normal(0, std, x.shape).astype(np.float32)
    x_noisy = np.clip(x + noise, 0, 1)
    x_noisy = (x_noisy * 255).astype(np.uint8)
    return Image.fromarray(x_noisy)


def salt_pepper_noise(img, severity=1):
    levels = [0.01, 0.03, 0.05, 0.08, 0.12]
    amount = levels[severity - 1]

    x = np.array(img).copy()
    h, w, c = x.shape

    num_pixels = int(amount * h * w)

    # salt
    coords = (np.random.randint(0, h, num_pixels), np.random.randint(0, w, num_pixels))
    x[coords[0], coords[1], :] = 255

    # pepper
    coords = (np.random.randint(0, h, num_pixels), np.random.randint(0, w, num_pixels))
    x[coords[0], coords[1], :] = 0

    return Image.fromarray(x)


def gaussian_blur(img, severity=1):
    levels = [0.5, 1.0, 1.5, 2.0, 2.5]
    radius = levels[severity - 1]
    return img.filter(ImageFilter.GaussianBlur(radius=radius))


def brightness(img, severity=1):
    levels = [0.8, 0.6, 0.5, 0.4, 0.3]  # darker
    factor = levels[severity - 1]
    return ImageEnhance.Brightness(img).enhance(factor)


def cutout(img, severity=1):
    levels = [6, 10, 14, 18, 22]  # patch size
    size = levels[severity - 1]

    x = np.array(img).copy()
    h, w, c = x.shape

    y = np.random.randint(0, h)
    x0 = np.random.randint(0, w)

    y1 = np.clip(y - size // 2, 0, h)
    y2 = np.clip(y + size // 2, 0, h)
    x1 = np.clip(x0 - size // 2, 0, w)
    x2 = np.clip(x0 + size // 2, 0, w)

    x[y1:y2, x1:x2, :] = 0
    return Image.fromarray(x)


CORRUPTION_FUNCS = {
    "gaussian_noise": gaussian_noise,
    "salt_pepper": salt_pepper_noise,
    "gaussian_blur": gaussian_blur,
    "brightness": brightness,
    "cutout": cutout,
}


In [3]:
import sys
sys.path.append(r"C:\Users\nigel\Desktop\prax\robust-anomaly-cifar10")

import src.dataset as d
print(dir(d))


['CIFAR10', 'CORRUPTION_FUNCS', 'CorruptedCIFAR10', 'Dataset', 'Image', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'torch']


In [4]:
import torch
from torch.utils.data import Dataset
from torchvision.datasets import CIFAR10
from PIL import Image


from src.corruptions import CORRUPTION_FUNCS
from src.dataset import CorruptedCIFAR10



class CorruptedCIFAR10(Dataset):
    """
    Returns:
      clean_img, corrupted_img, label
    """

    def __init__(
        self,
        root="./data/raw",
        train=True,
        download=True,
        transform=None,
        corruption_type=None,
        severity=1,
        corruption_prob=1.0,
    ):
        self.dataset = CIFAR10(root=root, train=train, download=download)
        self.transform = transform

        self.corruption_type = corruption_type
        self.severity = severity
        self.corruption_prob = corruption_prob

        if corruption_type is not None:
            if corruption_type not in CORRUPTION_FUNCS:
                raise ValueError(
                    f"Unknown corruption_type={corruption_type}. "
                    f"Choose from {list(CORRUPTION_FUNCS.keys())}"
                )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]

        # Ensure PIL image
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)

        clean_img = img

        corrupted_img = img
        if self.corruption_type is not None:
            if torch.rand(1).item() < self.corruption_prob:
                corrupted_img = CORRUPTION_FUNCS[self.corruption_type](
                    img, severity=self.severity
                )

        if self.transform is not None:
            clean_img = self.transform(clean_img)
            corrupted_img = self.transform(corrupted_img)

        return clean_img, corrupted_img, label
